INSTALL LIBRARY


In [ ]:
!pip install yfinance torch torchdiffeq scikit-learn

IMPORT DAN SET DEVICE

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random

from sklearn.preprocessing import StandardScaler
from torchdiffeq import odeint

note : mengimpor seluruh library yang diperlukan. NumPy digunakan untuk perhitungan numerik, Pandas untuk mengolah data, yfinance untuk mengambil data historis saham dari Yahoo Finance, dan Matplotlib untuk membuat visualisasi. Selanjutnya, StandardScaler digunakan untuk normalisasi data, sedangkan MSE dan MAE digunakan sebagai metrik evaluasi model. PyTorch dan torch.nn digunakan untuk membangun model Neural Network, sementara odeint dari torchdiffeq digunakan untuk menyelesaikan persamaan diferensial pada Neural ODE. Terakhir, program memeriksa ketersediaan GPU (CUDA) atau CPU agar proses komputasi dapat dijalankan pada perangkat yang paling sesuai.

PENGATURAN RANDOM SEED

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cpu")

note: Kode ini menetapkan random seed sebesar 42 agar hasil program tetap konsisten setiap kali dijalankan. Seed diterapkan pada random, NumPy, dan PyTorch, termasuk pada GPU (CUDA) jika tersedia. Selain itu, pengaturan deterministic=True dan benchmark=False digunakan agar proses komputasi lebih stabil dan dapat direproduksi. Terakhir, program menampilkan nilai random seed yang digunakan.

DOWNLOAD DATA BBCA

In [ ]:
ticker = "BBCA.JK"

data = yf.download(
    ticker,
    start="2015-01-01",
    end="2026-07-31",
    auto_adjust=False
)

data = data[['Close']]
data.dropna(inplace=True)

print("Jumlah data :", len(data))
display(data.head())

[*********************100%***********************]  1 of 1 completed

Jumlah data : 2834


Price,Close
Ticker,BBCA.JK
Date,
2015-01-02,2645.0
2015-01-05,2640.0
2015-01-06,2620.0
2015-01-07,2625.0
2015-01-08,2595.0


note : harga saham memiliki tren meningkat, namun juga mengalami fluktuasi yang cukup tinggi.
Hal ini menunjukkan bahwa data bersifat non-stasioner, karena memiliki tren dan variansi yang tidak konstan.


PREPROCESSING

note : Transformasi log-return dilakukan untuk menghilangkan tren dan membuat data lebih stabil secara statistik.

In [ ]:
# MENGHITUNG LOG RETURN
data["LogReturn"] = np.log(
    data["Close"] /
    data["Close"].shift(1)
)

data.dropna(inplace=True)

display(data.head())

Price,Close,LogReturn
Ticker,BBCA.JK,
Date,,
2015-01-05,2640.0,-0.001892
2015-01-06,2620.0,-0.007605
2015-01-07,2625.0,0.001907
2015-01-08,2595.0,-0.011494
2015-01-09,2585.0,-0.003861


note :Kode ini menghitung log-return menggunakan logaritma natural dari perbandingan harga penutupan hari ini (Close) dengan hari sebelumnya (Close.shift(1)), lalu menyimpannya pada kolom LogReturn. Selanjutnya, data.dropna(inplace=True) menghapus baris yang bernilai kosong (NaN), karena data pertama tidak memiliki harga penutupan hari sebelumnya sehingga log-return tidak dapat dihitung.
data berfluktuasi di sekitar nol dan tidak memiliki tren, sehingga lebih cocok untuk pemodelan matematis.

PARAMETER PENELITIAN

In [ ]:
# tanggal awal prediksi
tanggal_prediksi = "2026-06-29"

# jumlah data historis
JUMLAH_HARI = 5

MENENTUKAN DATA INPUT DAN TARGET

In [ ]:
idx = data.index.get_loc(tanggal_prediksi)

# data historis untuk training
data_input = data.iloc[idx-JUMLAH_HARI:idx].copy()

# data aktual yang akan dibandingkan
data_target = data.iloc[idx:idx+5].copy()

harga_aktual = data_target["Close"].values.flatten()

P0_aktual = data.iloc[idx-1]["Close"].item()

print("Rentang Data Historis")
print(data_input.index[0], "sampai", data_input.index[-1])

print("\nPeriode Prediksi")
print(data_target.index[0], "sampai", data_target.index[-1])

Rentang Data Historis
2026-06-22 00:00:00 sampai 2026-06-26 00:00:00

Periode Prediksi
2026-06-29 00:00:00 sampai 2026-07-03 00:00:00


STANDARDSCALER

In [ ]:
# =====================================================
# STANDARDISASI DATA
# =====================================================

scaler = StandardScaler()

# Fit hanya pada data historis
scaler.fit(data_input[['LogReturn']])

# Transform data historis
train_data_scaled = scaler.transform(data_input[['LogReturn']])

# Simpan parameter scaler
mu = scaler.mean_[0]
sigma = scaler.scale_[0]

print("Rata-rata (μ) :", mu)
print("Standar Deviasi (σ) :", sigma)

Rata-rata (μ) : -0.004008150176689212
Standar Deviasi (σ) : 0.02150506825833545


MEMBUAT TENSOR

In [ ]:
train_tensor = torch.tensor(
    train_data_scaled,
    dtype=torch.float32
).to(device)

# Nilai awal berasal dari log-return terakhir data historis
y0_raw = data_input.iloc[-1]["LogReturn"]

y0_scaled = scaler.transform([[y0_raw]])[0][0]

test_tensor = torch.tensor(
    [y0_scaled],
    dtype=torch.float32
).to(device)

# Harga penutupan terakhir sebagai harga awal prediksi
P0_aktual = data_input.iloc[-1]["Close"]

print("y0 (original):", y0_raw)
print("y0 (scaled):", y0_scaled)
print("Harga Awal (P0):", P0_aktual)

ValueError: Found array with dim 3. StandardScaler expected <= 2.

MODEL NEURAL ODE

In [ ]:
# MEMBANGUN MODEL NEURAL ODE
class ODEFunc(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(1,10),

            nn.Tanh(),

            nn.Linear(10,1)

        )

    def forward(self,t,y):
        return self.net(y)

TRAINING NEURAL ODE

In [ ]:
# =====================================================
# TRAINING MODEL
# =====================================================

set_seed(42)

func = ODEFunc().to(device)

optimizer = torch.optim.Adam(
    func.parameters(),
    lr=0.01
)

loss_fn = nn.MSELoss()

t_train = torch.linspace(
    0,
    1,
    len(train_tensor)
).to(device)

epochs = 300

loss_history = []

for epoch in range(epochs):

    optimizer.zero_grad()

    pred_y = odeint(
        func,
        train_tensor[0],
        t_train
    )

    loss = loss_fn(
        pred_y.squeeze(),
        train_tensor.squeeze()
    )

    loss.backward()

    optimizer.step()

    loss_history.append(loss.item())

print("Training selesai")
print("Loss akhir :", loss_history[-1])

METODE EULER

In [ ]:
def euler_method(func, y0, t, verbose=False):
    y = [y0.view(1)]
    h = t[1] - t[0]

    if verbose:
        print("\n--- PROSES INTERNAL GRADIEN EULER ---")

    for i in range(1, len(t)):
        gradien = func(None, y[-1].view(1, 1))

        if verbose:
            print(f"Hari {i} -> Input y_{i-1}: {y[-1].item():.8f} | Gradien f(x_{i-1}, y_{i-1}): {gradien.item():.8f}")

        y_next = y[-1] + h * gradien
        y.append(y_next.view(1))

    return torch.stack(y)

In [ ]:
METODE RK4

In [ ]:
def rk4_method(func, y0, t, verbose=False):
    y = [y0.view(1)]
    h = t[1] - t[0]

    if verbose:
        print("\n--- PROSES INTERNAL KOEFISIEN RK4 ---")

    for i in range(1, len(t)):
        yi = y[-1].view(1, 1)

        k1 = func(None, yi)
        k2 = func(None, yi + (h / 2) * k1)
        k3 = func(None, yi + (h / 2) * k2)
        k4 = func(None, yi + h * k3)

        if verbose:
            print(
                f"Hari {i} -> "
                f"k1: {k1.item():.8f} | "
                f"k2: {k2.item():.8f} | "
                f"k3: {k3.item():.8f} | "
                f"k4: {k4.item():.8f}"
            )

        y_next = yi + (h / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
        y.append(y_next.view(1))

    return torch.stack(y)

VALIDASI MODEL DENGAN MANUAL

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
from sklearn.preprocessing import StandardScaler
from torchdiffeq import odeint

# =====================================================================
# FUNGSI KUNCI SEED (WAJIB ADA DI DALAM SEL JIKA INGIN HASIL KONSISTEN)
# =====================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Memastikan operasi pada GPU/CPU bersifat deterministik
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# PANGGIL SEED DI SINI SEBELUM PROSES APAPUN DIMULAI
set_seed(42)

device = torch.device('cpu')
ticker = "BBCA.JK"

# 1. DOWNLOAD DATA HISTORIS BBCA
data = yf.download(
    ticker,
    start="2026-06-15",
    end="2026-07-10",
    auto_adjust=False
)

data = data[['Close']]
data.dropna(inplace=True)

# 2. HITUNG LOG-RETURN
data['LogReturn'] = np.log(data['Close'] / data['Close'].shift(1))
data.dropna(inplace=True)

# 3. SETTING FIT SCALER HANYA PADA DATA HISTORIS (22 Juni - 26 Juni)
data_input_mingguan = data.loc["2026-06-22":"2026-06-26"]

scaler = StandardScaler()
scaler.fit(data_input_mingguan[['LogReturn']])

# 4. AMBIL DATA TARGET AKTUAL (29 Juni - 3 Juli 2026) UNTUK EVALUASI PRESISI
data_target_mingguan = data.loc["2026-06-29":"2026-07-03"]
harga_aktual_target = data_target_mingguan['Close'].values.flatten().tolist()

# 5. HARGA AWAL AKTUAL (P0) = Harga Close hari Jumat, 26 Juni 2026
P0_aktual = data.loc["2026-06-26"]['Close'].item()

# 6. MENYIAPKAN TENSOR AWAL (y0) DARI HARI JUMAT 26 JUNI
y0_raw = data.loc["2026-06-26"]['LogReturn'].item()
y0_scaled = scaler.transform([[y0_raw]])[0][0]
test_tensor = torch.tensor([y0_scaled], dtype=torch.float32).to(device)


# =====================================================================
# BLOK TRAINING (Kunci seed dipanggil lagi di sini agar bobot awal NN tetap)
# =====================================================================
train_data_scaled = scaler.transform(data_input_mingguan[['LogReturn']])
train_tensor = torch.tensor(train_data_scaled, dtype=torch.float32).to(device)

if 'ODEFunc' not in globals():
    class ODEFunc(nn.Module):
        def __init__(self):
            super(ODEFunc, self).__init__()
            self.net = nn.Sequential(
                nn.Linear(1, 10),
                nn.Tanh(),
                nn.Linear(10, 1)
            )
        def forward(self, t, y):
            return self.net(y)

# PANGGIL ULANG SEED TEPAT SEBELUM INSTANSIASI MODEL
set_seed(42)

func = ODEFunc().to(device)
optimizer = torch.optim.Adam(func.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
t_train = torch.linspace(0, 1, len(train_tensor)).to(device)

epochs = 300
for epoch in range(epochs):
    optimizer.zero_grad()
    pred_y = odeint(func, train_tensor[0], t_train)
    loss = loss_fn(pred_y.squeeze(), train_tensor.squeeze())
    loss.backward()
    optimizer.step()

print("Training selesai khusus data 22-26 Juni (Hasil Terkunci).")


# =====================================================================
# DEFINISI SOLVER NUMERIK
# =====================================================================
# =====================================================================
# DEFINISI SOLVER NUMERIK (SUDAH DILENGKAPI OUTPUT GRADIEN TIAP HARI)
# =====================================================================
def euler_method(func, y0, t):
    y = [y0.view(1)]
    h = t[1] - t[0]
    print("\n--- PROSES INTERNAL GRADIEN EULER ---")
    for i in range(1, len(t)):
        # Menghitung gradien f(x, y) melalui Jaringan Syaraf
        gradien = func(None, y[-1].view(1, 1))
        print(f"Hari {i} -> Input y_{i-1}: {y[-1].item():.8f} | Gradien f(x_{i-1}, y_{i-1}): {gradien.item():.8f}")

        y_next = y[-1] + h * gradien
        y.append(y_next.view(1))
    return torch.stack(y)

def rk4_method(func, y0, t):
    y = [y0.view(1)]
    h = t[1] - t[0]
    print("\n--- PROSES INTERNAL KOEFISIEN RK4 ---")
    for i in range(1, len(t)):
        yi = y[-1].view(1, 1)
        k1 = func(None, yi)
        k2 = func(None, yi + (h/2) * k1)
        k3 = func(None, yi + (h/2) * k2)
        k4 = func(None, yi + h * k3)

        print(f"Hari {i} -> k1: {k1.item():.8f} | k2: {k2.item():.8f} | k3: {k3.item():.8f} | k4: {k4.item():.8f}")

        y_next = yi + (h/6) * (k1 + 2*k2 + 2*k3 + k4)
        y.append(y_next.view(1))
    return torch.stack(y)


# =====================================================================
# KODE VALIDASI OUTPUT & KONVERSI RUPIAH SECARA EKSPLISIT
# =====================================================================
print("\n======= A. ANGKA UNTUK HITUNGAN MANUAL DI KERTAS =======")
h = (t_eval[1]-t_eval[0]).item()
print(f"Ukuran Langkah (h)                       : {h:.2f}")
mu_internal = scaler.mean_[0]
sigma_internal = scaler.scale_[0]
print(f"Rata-rata Data Training (mu)             : {mu_internal:.8f}")
print(f"Standar Deviasi Data Training (sigma)    : {sigma_internal:.8f}")
print("-" * 55)

print(f"Nilai Awal Log-Return Scaled (y0_scaled) : {test_tensor[0].item():.8f}")
y0_original = scaler.inverse_transform([[test_tensor[0].item()]])[0][0]
print(f"Nilai Awal Log-Return Asli (y0_original) : {y0_original:.8f}")

func.eval()
with torch.no_grad():
    f_y0 = func(None, test_tensor[0].view(1, 1)).item()
print(f"Nilai Gradien Model f(y0, t0) Scaled    : {f_y0:.8f}")


print("\n======= B. CONVERSION LOG-RETURN TO RUPIAH (PREDIKSI 1 MINGGU) =======")
t_eval = torch.linspace(0, 1, 5).to(device)

with torch.no_grad():
    y_pred_euler_scaled = euler_method(
        func,
        test_tensor[0],
        t_eval,
        verbose=True
    )

    y_pred_rk4_scaled = rk4_method(
        func,
        test_tensor[0],
        t_eval,
        verbose=True
    )

log_ret_euler = scaler.inverse_transform(y_pred_euler_scaled.cpu().numpy().reshape(-1, 1)).flatten()
log_ret_rk4 = scaler.inverse_transform(y_pred_rk4_scaled.cpu().numpy().reshape(-1, 1)).flatten()

harga_pred_euler = P0_aktual * np.exp(log_ret_euler)
harga_pred_rk4 = P0_aktual * np.exp(log_ret_rk4)

print(f"Harga Dasar Bursa P0 (26 Juni) : Rp {P0_aktual:.2f}\n")
print("Hari ke- | Prediksi Euler (Rp) | Prediksi RK4 (Rp) | Harga Aktual (Rp)")
print("---------------------------------------------------------------------")
for i in range(5):
    print(f"   {i+1}     |    {harga_pred_euler[i]:.2f}     |    {harga_pred_rk4[i]:.2f}    |    {harga_aktual_target[i]:.2f}")


print("\n======= C. ANALISIS PRESISI MODEL (EVALUASI MAPE) =======")
mape_euler = np.mean(np.abs((np.array(harga_aktual_target) - harga_pred_euler) / np.array(harga_aktual_target))) * 100
mape_rk4 = np.mean(np.abs((np.array(harga_aktual_target) - harga_pred_rk4) / np.array(harga_aktual_target))) * 100

print(f"Rata-rata Error Prediksi Euler : {mape_euler:.2f}% (Tingkat Akurasi: {100-mape_euler:.2f}%)")
print(f"Rata-rata Error Prediksi RK4   : {mape_rk4:.2f}% (Tingkat Akurasi: {100-mape_rk4:.2f}%)")

FUNGSI PREDIKSI OTOMATIS

In [ ]:
# =====================================================
# FUNGSI PREDIKSI BERDASARKAN RENTANG HISTORIS
# =====================================================

def prediksi_model(jumlah_hari, tampilkan_detail=False):

    print("="*70)
    print(f"RENTANG HISTORIS : {jumlah_hari} HARI")
    print("="*70)

    # -------------------------------------------------
    # Menentukan data historis
    # -------------------------------------------------

    idx = data.index.get_loc(tanggal_prediksi)

    data_input = data.iloc[idx-jumlah_hari:idx].copy()

    data_target = data.iloc[idx:idx+5].copy()

    harga_aktual = data_target["Close"].values.flatten()

    P0 = data_input.iloc[-1]["Close"]

    # -------------------------------------------------
    # StandardScaler
    # -------------------------------------------------

    scaler = StandardScaler()

    scaler.fit(data_input[['LogReturn']])

    train_scaled = scaler.transform(data_input[['LogReturn']])

    train_tensor = torch.tensor(
        train_scaled,
        dtype=torch.float32
    ).to(device)

    y0_raw = data_input.iloc[-1]["LogReturn"]

    y0_scaled = scaler.transform([[y0_raw]])[0][0]

    test_tensor = torch.tensor(
        [y0_scaled],
        dtype=torch.float32
    ).to(device)

    # -------------------------------------------------
    # Training model baru
    # -------------------------------------------------

    set_seed(42)

    func = ODEFunc().to(device)

    optimizer = torch.optim.Adam(
        func.parameters(),
        lr=0.01
    )

    loss_fn = nn.MSELoss()

    t_train = torch.linspace(
        0,
        1,
        len(train_tensor)
    ).to(device)

    for epoch in range(300):

        optimizer.zero_grad()

        pred_y = odeint(
            func,
            train_tensor[0],
            t_train
        )

        loss = loss_fn(
            pred_y.squeeze(),
            train_tensor.squeeze()
        )

        loss.backward()

        optimizer.step()

    # -------------------------------------------------
    # Prediksi
    # -------------------------------------------------

    t_eval = torch.linspace(0,1,5).to(device)

    with torch.no_grad():

        y_euler = euler_method(
            func,
            test_tensor[0],
            t_eval
        )

        y_rk4 = rk4_method(
            func,
            test_tensor[0],
            t_eval
        )

    # -------------------------------------------------
    # Konversi ke Rupiah
    # -------------------------------------------------

    log_euler = scaler.inverse_transform(
        y_euler.cpu().numpy().reshape(-1,1)
    ).flatten()

    log_rk4 = scaler.inverse_transform(
        y_rk4.cpu().numpy().reshape(-1,1)
    ).flatten()

    harga_euler = P0*np.exp(log_euler)

    harga_rk4 = P0*np.exp(log_rk4)

    # -------------------------------------------------
    # MAPE
    # -------------------------------------------------

    mape_euler = np.mean(
        np.abs(
            (harga_aktual-harga_euler)
            /harga_aktual
        )
    )*100

    mape_rk4 = np.mean(
        np.abs(
            (harga_aktual-harga_rk4)
            /harga_aktual
        )
    )*100

    # -------------------------------------------------
    # OUTPUT
    # -------------------------------------------------

    if tampilkan_detail:

        print()

        print("Hari | Euler | RK4 | Aktual")

        print("-"*55)

        for i in range(5):

            print(
                f"{i+1:>4} | "
                f"{harga_euler[i]:>8.2f} | "
                f"{harga_rk4[i]:>8.2f} | "
                f"{harga_aktual[i]:>8.2f}"
            )

        print()

    print(f"MAPE Euler : {mape_euler:.2f}%")

    print(f"MAPE RK4   : {mape_rk4:.2f}%")

    return {

        "Rentang":jumlah_hari,

        "MAPE Euler":mape_euler,

        "MAPE RK4":mape_rk4

    }

PREDIKSI DENGAN RENTANG WAKTU 1 MINGGU

In [ ]:
hasil_1minggu = prediksi_model(
    jumlah_hari=5,
    tampilkan_detail=True
)

PREDIKSI DENGAN RENTANG WAKTU 2 MINGGU

In [ ]:
hasil_2minggu = prediksi_model(
    jumlah_hari=10
)

PREDIKSI DENGAN RENTANG WAKTU 1 BULAN

In [ ]:
hasil_1bulan = prediksi_model(
    jumlah_hari=20
)

PREDIKSI DENGAN RENTANG WAKTU 3 BULAN

In [ ]:
hasil_3bulan = prediksi_model(
    jumlah_hari=63
)

PREDIKSI DENGAN RENTANG WAKTU 6  BULAN


In [ ]:
hasil_6bulan = prediksi_model(
    jumlah_hari=126
)

PREDIKSI DENGAN RENTANG WAKTU 1 TAHUN

In [ ]:
hasil_1tahun = prediksi_model(
    jumlah_hari=252
)

PERBANDINGAN HASIL PREDISKSI DENGAN RENTANG WATU BERBEDA

In [ ]:
hasil = pd.DataFrame([

    hasil_1minggu,

    hasil_2minggu,

    hasil_1bulan,

    hasil_3bulan,

    hasil_6bulan,

    hasil_1tahun

])

hasil

RINGKASAN HASIL

In [ ]:
print("\n==============================")
print("RINGKASAN HASIL PENELITIAN")
print("==============================")

print("MSE  :", mse)
print("RMSE :", rmse)
print("MAE  :", mae)

print("\nEstimasi Orde Euler :", order_euler)
print("Estimasi Orde RK4   :", order_rk4)

print("\nError Euler :", errors_euler)
print("Error RK4   :", errors_rk4)

note: Untuk analisis numerik, metode Euler memiliki estimasi orde yang mendekati 1, sesuai dengan teori bahwa Euler berorde satu.
Sedangkan RK4 menghasilkan nilai orde yang tidak stabil karena error sangat kecil mendekati nol, namun secara teori RK4 berorde empat.

Dari sisi error, terlihat bahwa error Euler masih lebih besar meskipun menurun seiring bertambahnya step, sedangkan RK4 memiliki error yang jauh lebih kecil bahkan mendekati nol.

Secara keseluruhan, hasil ini menunjukkan bahwa RK4 lebih akurat, lebih cepat konvergen, dan lebih stabil dibandingkan metode Euler dalam menyelesaikan Neural ODE.
